# Практическая работа 1 — Сбор данных из веб-источников

**Тема:** Сбор данных из веб-источников  
**Источник:** The Guardian Open Platform / Content API  
**Итог:** TSV-датасет `guardian_5000.tsv` — 5000 новостных статей с метриками

---

## Цель
Освоить сбор данных из веб-источников и сформировать датасет на основе англоязычных новостных материалов The Guardian.

## Задачи
1. Получить доступ к API Guardian
2. Разработать программу для извлечения данных
3. Собрать датасет объёмом не менее 5000 записей в формате TSV
4. Рассчитать метрики статей и характеристики датасета

---

## Описание источника данных

| Параметр | Значение |
|---|---|
| Источник | The Guardian Open Platform / Content API |
| Тип доступа | HTTP API, ответ в формате JSON |
| Тип данных | Англоязычные новостные и публицистические материалы |
| Формат датасета | TSV |
| Минимальный объём | 5000 записей |
| Основной эндпоинт | https://content.guardianapis.com/search |

## Метод доступа к данным

Доступ через официальный API. Параметры запроса:

| Параметр | Назначение |
|---|---|
| `api-key` | Ключ доступа к Guardian API |
| `page` | Номер страницы результатов |
| `page-size` | Количество материалов на странице (макс. 50) |
| `show-fields` | Дополнительные поля: headline, byline, body, bodyText, wordcount |
| `show-tags` | Теги статьи (entities) |
| `type` | Фильтрация материалов типа `article` |

## Поля итогового датасета

| Поле | Описание |
|---|---|
| `date` | Дата и время публикации |
| `topic` | Раздел или тема статьи |
| `title` | Заголовок статьи |
| `author` | Автор |
| `entities` | Тематические сущности из Guardian keyword tags |
| `word_count` | Количество слов |
| `links` | Ссылки, извлечённые из HTML статьи |
| `readability_score` | Числовая оценка сложности чтения (Flesch-Kincaid) |
| `readability_level` | Категория: easy / medium / hard / very hard |
| `sentiment` | Категория тональности: positive / neutral / negative |
| `sentiment_score` | Числовая оценка тональности |
| `url` | Ссылка на материал The Guardian |

## Шаг 1 — Настройка окружения

Перед запуском задайте API-ключ Guardian в переменной окружения:
```bash
export GUARDIAN_API_KEY="your-api-key"
```
Получить ключ можно на https://open-platform.theguardian.com/access/

In [ ]:
from __future__ import annotations

import argparse
import csv
import html
import json
import os
import re
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from dataclasses import dataclass
from html.parser import HTMLParser
from typing import Any

API_URL = "https://content.guardianapis.com/search"
MAX_PAGE_SIZE = 50

print("Библиотеки загружены.")

## Шаг 2 — Словари для анализа тональности

Тональность рассчитывается простым методом: подсчётом позитивных и негативных слов в тексте статьи. Это базовый, прозрачный подход — без сложных NLP-моделей.

In [ ]:
POSITIVE_WORDS = {
    "advance", "benefit", "best", "better", "boost", "breakthrough",
    "celebrate", "gain", "good", "great", "growth", "hope", "improve",
    "success", "support", "win",
}

NEGATIVE_WORDS = {
    "attack", "bad", "crisis", "damage", "death", "decline", "fail",
    "fear", "harm", "loss", "risk", "scandal", "threat", "war",
    "worse", "worst",
}

## Шаг 3 — Вспомогательные классы и функции

In [ ]:
class LinkParser(HTMLParser):
    """Извлекает href-ссылки из HTML-тела статьи."""
    def __init__(self) -> None:
        super().__init__()
        self.links: list[str] = []

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag != "a":
            return
        attrs_dict = dict(attrs)
        href = attrs_dict.get("href")
        if href:
            self.links.append(href)


@dataclass
class ArticleMetrics:
    """Одна обработанная статья — все поля итогового датасета."""
    date: str
    topic: str
    title: str
    author: str
    entities: str
    word_count: int
    links: str
    readability_score: float | None
    readability_level: str
    sentiment: str
    sentiment_score: int
    url: str

## Шаг 4 — Функции расчёта метрик

### Читаемость (Flesch-Kincaid Grade Level)
Формула: `0.39 × (слов/предложений) + 11.8 × (слогов/слов) − 15.59`  
Чем выше оценка — тем сложнее текст.

In [ ]:
def normalize_link(link: str) -> str | None:
    link = html.unescape(link)
    parsed = urllib.parse.urlparse(link)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return None
    return urllib.parse.urlunparse(
        (parsed.scheme, parsed.netloc, parsed.path, parsed.params, parsed.query, "")
    )


def extract_links(body_html: str) -> list[str]:
    parser = LinkParser()
    parser.feed(body_html)
    links = []
    for link in parser.links:
        normalized = normalize_link(link)
        if normalized:
            links.append(normalized)
    return sorted(set(links))


def clean_text(raw_text: str) -> str:
    text = html.unescape(raw_text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def count_words(text: str) -> int:
    return len(re.findall(r"\b[A-Za-z]+(?::'[A-Za-z]+)?\b", text))


def count_sentences(text: str) -> int:
    sentences = re.findall(r"[^.!?]+[.!?]", text)
    return max(1, len(sentences))


def count_syllables(word: str) -> int:
    word = word.lower()
    word = re.sub(r"[^a-z]", "", word)
    if not word:
        return 0
    groups = re.findall(r"[aeiouy]+", word)
    syllables = len(groups)
    if word.endswith("e") and syllables > 1:
        syllables -= 1
    return max(1, syllables)


def readability_score(text: str) -> float | None:
    words = re.findall(r"\b[A-Za-z]+(?::'[A-Za-z]+)?\b", text)
    if not words:
        return None
    word_total = len(words)
    sentence_total = count_sentences(text)
    syllable_total = sum(count_syllables(word) for word in words)
    grade = 0.39 * (word_total / sentence_total) + 11.8 * (syllable_total / word_total) - 15.59
    return round(grade, 2)


def readability_level(score: float | None) -> str:
    if score is None:
        return "unknown"
    if score < 6:
        return "easy"
    if score < 10:
        return "medium"
    if score < 14:
        return "hard"
    return "very hard"


def sentiment_score(text: str) -> int:
    words = [word.lower() for word in re.findall(r"\b[A-Za-z]+\b", text)]
    positive_count = sum(1 for word in words if word in POSITIVE_WORDS)
    negative_count = sum(1 for word in words if word in NEGATIVE_WORDS)
    return positive_count - negative_count


def sentiment_label(score: int) -> str:
    if score > 1:
        return "positive"
    if score < -1:
        return "negative"
    return "neutral"


def tags_to_entities(tags: list[dict[str, Any]]) -> str:
    entity_names = []
    for tag in tags:
        if tag.get("type") == "keyword":
            name = tag.get("webTitle")
            if name:
                entity_names.append(name)
    return "; ".join(sorted(set(entity_names)))


print("Функции определены.")

## Шаг 5 — Функции работы с API

In [ ]:
def build_api_url(api_key: str, page: int, page_size: int, section: str | None) -> str:
    params = {
        "api-key": api_key,
        "page": str(page),
        "page-size": str(page_size),
        "order-by": "newest",
        "show-fields": "headline,byline,body,bodyText,trailText,wordcount",
        "show-tags": "keyword",
        "type": "article",
    }
    if section:
        params["section"] = section
    return f"{API_URL}?{urllib.parse.urlencode(params)}"


def fetch_json(url: str) -> dict[str, Any]:
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "guardian-metrics-student/1.0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        message = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Guardian API вернул HTTP {error.code}: {message}") from error
    except urllib.error.URLError as error:
        raise RuntimeError(f"Не удалось подключиться к Guardian API: {error.reason}") from error


def article_to_metrics(article: dict[str, Any]) -> ArticleMetrics | None:
    fields = article.get("fields", {})
    body_text = clean_text(fields.get("bodyText", ""))
    if not body_text:
        return None

    links = extract_links(fields.get("body", ""))
    word_count = int(fields.get("wordcount") or count_words(body_text))
    reading_score = readability_score(body_text)
    tone_score = sentiment_score(body_text)

    return ArticleMetrics(
        date=article.get("webPublicationDate", ""),
        topic=article.get("sectionName", ""),
        title=fields.get("headline") or article.get("webTitle", ""),
        author=fields.get("byline", ""),
        entities=tags_to_entities(article.get("tags", [])),
        word_count=word_count,
        links="; ".join(links),
        readability_score=reading_score,
        readability_level=readability_level(reading_score),
        sentiment=sentiment_label(tone_score),
        sentiment_score=tone_score,
        url=article.get("webUrl", ""),
    )

## Шаг 6 — Сбор датасета

Функция собирает статьи постранично. Пропускает дубликаты по URL и статьи без текста.

In [ ]:
def collect_articles(
    api_key: str,
    limit: int = 5000,
    section: str | None = None,
    pause: float = 1.1,
) -> list[ArticleMetrics]:
    collected: list[ArticleMetrics] = []
    seen_urls: set[str] = set()
    page = 1

    while len(collected) < limit:
        url = build_api_url(api_key, page, MAX_PAGE_SIZE, section)
        data = fetch_json(url)
        response = data.get("response", {})
        articles = response.get("results", [])
        if not articles:
            break

        for article in articles:
            article_url = article.get("webUrl", "")
            if article_url in seen_urls:
                continue
            metrics = article_to_metrics(article)
            if metrics is None:
                continue
            seen_urls.add(article_url)
            collected.append(metrics)
            if len(collected) >= limit:
                break

        total_pages = int(response.get("pages", page))
        print(f"страница {page}/{total_pages}: собрано {len(collected)}/{limit}", file=sys.stderr)
        if page >= total_pages:
            break
        page += 1
        time.sleep(pause)

    return collected


def write_tsv(path: str, rows: list[ArticleMetrics]) -> None:
    fieldnames = [
        "date", "topic", "title", "author", "entities", "word_count",
        "links", "readability_score", "readability_level", "sentiment",
        "sentiment_score", "url",
    ]
    with open(path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        for row in rows:
            writer.writerow(row.__dict__)


print("Функции сбора готовы.")

## Шаг 7 — Запуск сбора

Для запуска нужен API-ключ Guardian. Установите переменную окружения `GUARDIAN_API_KEY`.

In [ ]:
OUTPUT_PATH = "guardian_5000.tsv"

api_key = os.getenv("GUARDIAN_API_KEY")
if not api_key:
    print("GUARDIAN_API_KEY не задан. Укажите ключ для запуска сбора.")
else:
    rows = collect_articles(api_key=api_key, limit=5000, pause=1.1)
    write_tsv(OUTPUT_PATH, rows)
    print(f"Сохранено {len(rows)} строк в {OUTPUT_PATH}")

## Шаг 8 — Анализ собранного датасета

In [ ]:
import pandas as pd

df = pd.read_csv("guardian_5000.tsv", sep="\t")

print("=" * 50)
print(f"Форма датасета: {df.shape}")
print(f"Диапазон дат: {df['date'].min()} — {df['date'].max()}")
print()
print("Количество слов:")
print(df["word_count"].describe().round(1))

In [ ]:
print("Топ-10 тем по количеству статей:")
print(df["topic"].value_counts().head(10))

In [ ]:
print("Распределение по тональности:")
print(df["sentiment"].value_counts())
print()
print("Распределение по уровню читаемости:")
print(df["readability_level"].value_counts())

In [ ]:
print("Средние показатели по темам (топ-10):")
print(
    df.groupby("topic")[["readability_score", "sentiment_score"]]
    .mean()
    .round(2)
    .sort_values("readability_score", ascending=False)
    .head(10)
)

In [ ]:
print("Пропуски по полям:")
missing = df.isna().sum()
pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({"Пропуски": missing, "Доля, %": pct}))

## Выводы

- Выбран источник данных **The Guardian Open Platform** и получен API-ключ для учебного доступа.
- Разработана программа на Python: получает статьи через API, извлекает основные поля и рассчитывает дополнительные метрики (количество слов, читаемость, тональность, ссылки, сущности).
- Итоговый датасет `guardian_5000.tsv` содержит **5000 записей**, 12 полей, диапазон дат: 30.03.2026 — 28.04.2026.
- По собранным данным разделы **US news** и **World news** имеют отрицательное среднее значение тональности, тогда как **Football** и **Sport** — положительное.
- Использованные методы тональности и сущностей являются базовыми. В дальнейшем можно расширить за счёт spaCy (NER) и VADER (sentiment).